# Thermal-Aware Qubit Mapping Optimization
### ESP-Based Noise-Aware Qubit Mapping with an Excited-State Population ($p_1$) Thermal Term
**Author**: Aarush Mishra · SVKM's NMIMS MPSTME  
**Track**: Noise-Aware Quantum Computing · *Qiskit Fall Fest 2026*

---
## Overview
Superconducting transmon quantum processors are intrinsically subject to environmental thermal noise and non-equilibrium quasiparticle excitations. While standard compilers optimize for SWAP count or standard Estimated Success Probability (ESP), they routinely ignore spatial and temporal heterogeneity in physical qubit excited-state populations ($p_1$).

This interactive notebook demonstrates:
1. **Thermal noise modeling** and $p_1$ excitation calculation across physical qubits.
2. **Thermal-aware heuristic subgraph mapping** compared against Random and Qiskit Levels 1 & 3.
3. **Noisy Qiskit Aer simulations** under thermal relaxation and depolarizing errors.
4. **Live IBM Quantum hardware results** executed on the 156-qubit Heron r2 processor `ibm_marrakesh`.


In [ ]:
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import networkx as nx

# Add project root to path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(root_dir))

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeVigoV2

from src.benchmarks.circuits import get_all_benchmarks
from src.mappers.mappers import mapper_random, mapper_qiskit_default, mapper_esp
from src.metrics.esp import esp_standard, esp_thermal, get_p1
from src.noise_model.thermal import build_thermal_noise_model
from src.noise_model.profile import load_config

print("All modules loaded successfully.")


## 1. Hardware Topology & Thermal Profiling
We inspect the coupling graph and physical qubit properties ($T_1$, $T_2$, readout error $\epsilon_{ro}$, and excited-state population $p_1$) of a target quantum processor.


In [ ]:
backend = FakeVigoV2()
target = backend.target

print(f"Target Backend: {backend.name} ({target.num_qubits} qubits)")
qubit_data = []
for q in range(target.num_qubits):
    props = target.qubit_properties[q]
    t1_us = props.t1 * 1e6
    t2_us = props.t2 * 1e6
    ro_err = target['measure'][(q,)].error
    p1_val = get_p1(q, target, temps_mk={q: 15.0 + 10.0 * (q % 2)})
    qubit_data.append({
        'Qubit': q,
        'T1 (us)': round(t1_us, 1),
        'T2 (us)': round(t2_us, 1),
        'Readout Error': round(ro_err, 4),
        'Thermal p1': round(p1_val, 4)
    })

df_qubits = pd.DataFrame(qubit_data)
display(df_qubits)


## 2. Benchmark Circuits
We evaluate 4 canonical quantum algorithms:
1. **GHZ-5**: Highly sensitive to multi-qubit entanglement fidelity.
2. **Bernstein-Vazirani (BV-5)**: Tests phase-kickback oracle execution.
3. **Quantum Fourier Transform (QFT-5)**: Dense multi-angle controlled rotations.
4. **Mirror-5**: Heavy routing circuit with random CX layers inverted to $|0...0\rangle$.


In [ ]:
benchmarks = get_all_benchmarks(5)
for name, circ in benchmarks.items():
    print(f"Benchmark: {name:<8} | Qubits: {circ.num_qubits} | Depth: {circ.depth()} | Ops: {dict(circ.count_ops())}")


## 3. Qubit Mapping Strategies
We map the benchmark circuits onto the target topology using:
- **Random**: Random permutation baseline.
- **Qiskit Level 1**: Standard lightweight layout and routing.
- **Qiskit Level 3**: SabreLayout + SabreSwap with depth optimization.
- **Standard ESP**: Heuristic connected subgraph search optimizing gate + readout fidelities.
- **Thermal ESP**: Heuristic search penalizing excess excited-state populations ($p_{1, \text{excess}} = \max(0, p_1 - \epsilon_{ro})$).


In [ ]:
circuit_ghz = benchmarks['ghz']

# Run mappers
tc_rand, _, _, lay_rand, _ = mapper_random(circuit_ghz, target, num_draws=1, seed=42)[0]
tc_l1, _, _, lay_l1, _ = mapper_qiskit_default(circuit_ghz, target, level=1)
tc_l3, _, _, lay_l3, _ = mapper_qiskit_default(circuit_ghz, target, level=3)
tc_esp, _, _, lay_esp, _ = mapper_esp(circuit_ghz, target, use_thermal=False, exhaustive=False)
tc_th, _, _, lay_th, _ = mapper_esp(circuit_ghz, target, temps_mk={0: 45.0, 1: 15.0, 2: 15.0, 3: 45.0, 4: 15.0}, use_thermal=True, exhaustive=False)

mapping_summary = [
    {'Method': 'Random', 'Layout': str(lay_rand), '2Q Gates': tc_rand.count_ops().get('cx', tc_rand.count_ops().get('cz', 0)), 'Depth': tc_rand.depth()},
    {'Method': 'Qiskit L1', 'Layout': str(lay_l1), '2Q Gates': tc_l1.count_ops().get('cx', tc_l1.count_ops().get('cz', 0)), 'Depth': tc_l1.depth()},
    {'Method': 'Qiskit L3', 'Layout': str(lay_l3), '2Q Gates': tc_l3.count_ops().get('cx', tc_l3.count_ops().get('cz', 0)), 'Depth': tc_l3.depth()},
    {'Method': 'ESP (No Thermal)', 'Layout': str(lay_esp), '2Q Gates': tc_esp.count_ops().get('cx', tc_esp.count_ops().get('cz', 0)), 'Depth': tc_esp.depth()},
    {'Method': 'Thermal ESP', 'Layout': str(lay_th), '2Q Gates': tc_th.count_ops().get('cx', tc_th.count_ops().get('cz', 0)), 'Depth': tc_th.depth()},
]

df_maps = pd.DataFrame(mapping_summary)
display(df_maps)


## 4. Noisy Simulation under Thermal Noise Models
We construct an Aer thermal relaxation noise model with temperature-dependent $p_1$ population injection.


In [ ]:
temps_hot = {0: 45.0, 1: 15.0, 2: 15.0, 3: 45.0, 4: 15.0}
noise_model = build_thermal_noise_model(target, temps_mk=temps_hot)
sim = AerSimulator(noise_model=noise_model)

# Simulate GHZ across methods
sim_results = {}
for name, tc in [('Random', tc_rand), ('Qiskit L1', tc_l1), ('Qiskit L3', tc_l3), ('ESP', tc_esp), ('Thermal ESP', tc_th)]:
    counts = sim.run(tc, shots=4096).result().get_counts()
    p_succ = (counts.get('00000', 0) + counts.get('11111', 0)) / 4096.0
    sim_results[name] = p_succ

df_sim = pd.DataFrame(list(sim_results.items()), columns=['Method', 'Simulated Success Prob (GHZ)'])
display(df_sim)


## 5. Live IBM Quantum Hardware Results (`ibm_marrakesh`)
We inspect the experimental validation executed on IBM's 156-qubit Heron r2 quantum processor **`ibm_marrakesh`** (Job IDs: `db3uv1cvf2bc73ctqt1g` & `db3uvecvf2bc73ctqto0`).


In [ ]:
hw_csv_path = root_dir / 'results' / 'hardware' / 'runtime_table.csv'
if hw_csv_path.exists():
    df_hw = pd.read_csv(hw_csv_path)
    
    # Aggregate over repetitions
    grouped = df_hw.groupby(['benchmark', 'method'])['metric_val'].agg(['mean', 'std', 'count']).reset_index()
    grouped['95% CI'] = 1.96 * np.sqrt(grouped['mean'] * (1.0 - grouped['mean']) / 16384)
    display(grouped.head(12))
else:
    print("Hardware results CSV not found at:", hw_csv_path)


## 6. Real QPU Performance Visualization
Below is the comparison of real physical execution on `ibm_marrakesh` for GHZ-5.


In [ ]:
from IPython.display import Image, display

chart_img = root_dir / 'results' / 'figures' / 'slide11_hardware_results_chart.png'
circuit_img = root_dir / 'results' / 'figures' / 'slide10_transpiled_circuit.png'

if chart_img.exists():
    print("=== Slide 11: Real Hardware Results Bar Chart ===")
    display(Image(filename=str(chart_img)))

if circuit_img.exists():
    print("=== Slide 10: Transpiled Circuit on Heron r2 (Native CZ) ===")
    display(Image(filename=str(circuit_img)))


## 7. Key Findings & Scientific Takeaways
1. **Connectivity Search Protects Against Noisy Edges**:
   ESP outperforms random mapping by **+16.1%** on GHZ-5 ($90.3\%$ vs. $74.2\%$) and by **+23.7%** on Mirror-5 ($73.9\%$ vs. $50.2\%$) on `ibm_marrakesh`.
2. **Heron r2 Native CZ Basis**:
   The transpiler maps 2-qubit interactions directly into native Controlled-Z (`CZ`) gates. For GHZ-5, this achieves a compact circuit depth of 16 with only 4 CZ gates.
3. **Physical Thermal Convergence vs. Simulated Hot-Spots**:
   On physical QPUs, automated calibrations ensure that top-tier physical qubits have excited-state populations within the readout error margin ($P(1|0) \approx \epsilon_{ro}$), leading to $p_{1, \text{excess}} \approx 0$. In simulation with injected hot-spots, Thermal ESP achieves up to **+12%** fidelity gain by steering circuits away from high-noise qubits.
